# siRNA Patent Landscape Pipeline
### From EPO patent records to a structured table of siRNA activity data

**What this notebook does.** It finds patents about small interfering RNA (siRNA) at the European Patent Office (EPO), keeps the relevant ones, downloads their full text, and converts the experimental tables inside those patents into clean CSV files that are ready for analysis.

**Why it exists.** Patents hold a large amount of siRNA activity data (duplex sequences, cell lines, doses, percent knockdown, IC50 values) that never reaches public databases. The data sits inside long free-text documents and irregularly formatted tables. This pipeline automates the path from a patent number to one row per measurement.

**What you need.** An EPO OPS account (free tier), at least one Groq API key (free tier), and the `sirna_pipeline` package installed (Section 0).

**Never seen a patent database before?** The next cell is a one-page glossary. Every term it defines shows up later as a column name, a function argument or a filename.

### How to run it

1. Run the cells from top to bottom the first time. Each section reads the files written by the section before it.
2. Sections are also independent. If the input files are already in the working directory, any section can be run on its own.
3. All API keys are typed once, in the **Credentials** cell. No key is written into the notebook file.
4. Every filename passed between stages, and the pilot patent list, are set once in the **Run configuration** cell. No later section carries a hard-coded path.
5. Sections 4 to 7 are slow by design. EPO enforces an 8 second pause between requests, and the Groq free tier limits how fast the LLM calls can run.

### Pipeline at a glance

| Section | Stage | Module and entry point | Reads | Writes |
|---|---|---|---|---|
| 1 | Patent ID extraction | `epo.search.download_patent_ids`, once per strategy | EPO OPS (live) | `EPO_siRNA_IDs_<years>_codes_only.csv`, `..._terms_only.csv`, `..._only_applicant_Alnylam.csv` |
| 2 | Bibliographic metadata | `epo.biblio.fetch_biblio_from_csv` | one ID CSV from Section 1 | `<input name>_metadata.csv` |
| 3 | Tier filtering | `filtering.tiers.apply_filters` | the metadata CSV | `..._metadata_filtered.csv` |
| 4 | Full-text XML download | `epo.fulltext.download_eps_xmls_with_ops` | one ID CSV | `eps_xmls/*.xml`, `successful_downloads.csv`, `not_in_eps.csv` |
| 5 | Table isolation | `tables.isolate.extract_tables_from_patent` | `eps_xmls/*.xml` | `isolated_tables/*.xml` |
| 6 | XML to CSV, header cleaning (LLM) | `tables.parse.convert_directory` | `isolated_tables/` | `csv_output/*_tables.csv`, `csv_output/*_context.txt`, `csv_output/debug/` |
| 7 | Primary table assembly (LLM + DuckDB) | `assembly.build.build_primary_table` | `csv_output/` | `1_final_tables/`, `2_review/`, `3_per_file_drafts/`, `4_trace/` |

Sections 1 to 4 use the **EPO OPS** API. Sections 6 and 7 use the **Groq** LLM API. Sections 3 and 5 are local and free.

### Package layout

The pipeline is now an installable package under `src/sirna_pipeline/`, so no script has to sit next to this notebook. Install it once in editable mode (Section 0) and every import below resolves from anywhere.

```text
sirna_pipeline/
├── epo/
│   ├── search.py         Stage 1, IDs from codes, keywords or applicant name
│   ├── biblio.py         Stage 2, bibliographic metadata
│   └── fulltext.py       Stage 4, full-text XML download
├── filtering/
│   └── tiers.py          Stage 3, rule-based tier classification
├── tables/
│   ├── isolate.py        Stage 5, isolates the experimental tables
│   └── parse.py          Stage 6, table XML to CSV with LLM headers
└── assembly/
    ├── build.py          Stage 7 entry point
    ├── core.py           schemas, Groq client pool, validation
    ├── routing.py        decides what each table measures
    └── sql_builder.py    builds and runs the DuckDB SQL
```

## Patent vocabulary in one page

Read this once and the rest of the notebook reads normally. Everything here appears later as a column name, a function argument or a filename.

| Term | What it means here |
|---|---|
| **Publication** | One published document. Every `Patent_ID` in this notebook is a publication, not an invention |
| **Application vs grant** | The same invention is published at least twice: first as an application (kind codes `A1`, `A2`, `A3`, `A4`), later as a granted patent (`B1`, `B2`, `B3`). Claims are usually narrowed in between, so an `A` document is not a substitute for its `B`. Section 4 downloads both |
| **Kind code** | The letter and digit at the end of an identifier. `EP2723758B1` is a granted European patent |
| **DocDB identifier** | The `country + number + kind` format used throughout, for example `US7056704B2` |
| **Patent family** | The same invention filed in several countries, plus its applications and grants, all sharing one `Family_ID`. They describe the same experiments, so Section 1 keeps one member per family |
| **Applicant** | The company that filed the patent. Not the inventor, who is a person. One company files under several legal entity names, which is why Section 1c searches `"Alnylam*"` with a wildcard |
| **CPC and IPC** | Subject codes assigned by patent examiners, like library shelf marks. IPC is international, CPC is the more detailed EPO and USPTO extension. `C12N15/113` is the siRNA code |
| **Priority date** | When the invention was first filed anywhere. Use this one to order patents chronologically |
| **Publication date** | When this particular document became public, usually 18 months after the priority date. The EPO search field `pd=` filters on this one |
| **EXAMPLES section** | The part of the description holding the experiments. Section 5 uses this heading as the boundary between background and data |
| **CQL** | The query syntax the EPO search service accepts. Fields used here: `cpc=`, `ipc=`, `ta=` (title and abstract), `pa=` (applicant), `pd=` (publication date) |
| **OPS** | Open Patent Services, the EPO REST API. Sections 1, 2 and 4. Needs credentials, consumes the weekly 4 GB quota |
| **EPS** | European Publication Server, a separate open service holding EP full-text XML. Section 4. No credentials, no quota |
| **CALS table** | The XML table model patent documents use: column names in `<thead>`, data in `<tbody>`, cells that can span columns or rows. Section 6 exists mostly because drafters break these rules |

**Why patents at all.** A published patent must describe the invention well enough to reproduce it, so siRNA patents contain full experimental tables: duplex sequences, target genes, cell lines, doses, percent knockdown, IC50 values. Most of it never reaches a public database, and none of it is downloadable in structured form.


## 0. Environment setup

Run this once per environment. The package declares its own dependencies in `pyproject.toml`, so an editable install pulls in everything the pipeline needs.

| Package | Used for |
|---|---|
| `requests` | HTTP calls to the EPO OPS and European Publication Server (EPS) APIs |
| `pandas`, `numpy` | all tabular data handling |
| `beautifulsoup4`, `lxml` | XML parsing during table isolation and conversion |
| `groq` | client for the Groq LLM API (header repair and table mapping) |
| `duckdb` | local SQL engine that runs the LLM-generated `SELECT` queries |
| `openpyxl` | lets pandas read and write Excel files for manual inspection |
| `pyyaml` | reads the configuration files under `config/` |

The path below assumes this notebook sits in `notebooks/`, so `..` is the repository root. Change it if the notebook lives somewhere else.

The second cell is a smoke test: it imports every stage module, so a broken install shows up here rather than halfway through a run that has already spent API quota.


In [ ]:
# Editable install of the pipeline itself, plus every dependency in pyproject.toml.
# ".." is the repository root, since this notebook lives in notebooks/.
%pip install -e ..


In [ ]:
from importlib import import_module

for module in [
    "epo.search", "epo.biblio", "epo.fulltext",
    "filtering.tiers",
    "tables.isolate", "tables.parse",
    "assembly.build",
]:
    import_module(f"sirna_pipeline.{module}")

print("sirna_pipeline installed. All stage modules import correctly.")


## Credentials (run this first)

Every API key used in this notebook is set here, once, and reused by all later sections.

The cell reads environment variables first. If a variable is not set, it asks for the value with `getpass`, so the key is typed at run time and never saved inside the notebook file. That keeps the notebook safe to share or commit.

To skip the prompts, set the variables before starting Jupyter:

```bash
export EPO_CONSUMER_KEY=your_key
export EPO_CONSUMER_SECRET=your_secret
export GROQ_API_KEYS=key1,key2,key3
```

Two notes:

- EPO OPS keys come from the OPS developer portal. The free tier allows 4 GB of downloaded data per week.
- `GROQ_API_KEYS` accepts one key or several keys separated by commas, and the scripts rotate between them. Groq free-tier limits apply per account, so extra keys only raise throughput if they belong to different accounts.

In [2]:
import os
from getpass import getpass

def _credential(env_name, prompt):
    """Return a credential from the environment, or prompt for it (never stored)."""
    return os.getenv(env_name) or getpass(prompt)

# --- EPO OPS (Sections 1 to 4) ---
CONSUMER_KEY    = _credential("EPO_CONSUMER_KEY",    "EPO OPS consumer key: ")
CONSUMER_SECRET = _credential("EPO_CONSUMER_SECRET", "EPO OPS consumer secret: ")

# --- Groq (Sections 6 and 7). One or more keys, comma-separated. ---
# Groq free-tier limits are per ACCOUNT, so extra keys only raise throughput
# if they come from different Groq accounts.
GROQ_API_KEYS = _credential("GROQ_API_KEYS", "Groq API key(s), comma-separated: ")

print("Credentials loaded (EPO OPS + Groq).")


Credentials loaded (EPO OPS + Groq).


## Run configuration (run this second)

Every filename the pipeline hands from one stage to the next is set here, once.

Section 1 builds its own output name from the years it was given, so the years used there decide what the later sections have to read. `IDS_CSV` records that choice explicitly. It selects which Section 1 output the rest of the notebook consumes, and it does not have to be the search that ran most recently. `METADATA_CSV` and `FILTERED_CSV` are derived from it, so Sections 2 and 3 cannot drift out of step.

`PILOT` is the shared list of test patents. Section 5 uses it as filenames and Section 7 as file prefixes, so the two lists can no longer disagree.


In [ ]:
# --- Which Section 1 output the later stages consume ---------------------
IDS_CSV     = "EPO_siRNA_IDs_2022_2025_terms_only.csv"               # written by 1b
IDS_ALNYLAM = "EPO_siRNA_IDs_2022_2025_only_applicant_Alnylam.csv"   # written by 1c

# Derived names, so Sections 2 and 3 always match what was actually written.
METADATA_CSV = IDS_CSV.replace(".csv", "_metadata.csv")        # Section 2 writes this
FILTERED_CSV = METADATA_CSV.replace(".csv", "_filtered.csv")   # Section 3 writes this

# --- Folders -------------------------------------------------------------
XML_DIR    = "eps_xmls"           # Section 4 writes, Section 5 reads
TABLE_DIR  = "isolated_tables"    # Section 5 writes, Section 6 reads
CSV_DIR    = "csv_output"         # Section 6 writes, Section 7 reads
OUTPUT_CSV = "primary_table.csv"  # Section 7: its folder becomes the run root

# --- Pilot set: ten patents covering different table layouts -------------
# Section 5 appends ".xml" to these; Section 7 uses them as file prefixes.
PILOT = [
    "EP2723758NWB1",   # 1
    "EP2999785NWB1",   # 2
    "EP3146049NWB1",   # 3
    "EP3329924NWA1",   # 4
    "EP3872179NWA1",   # 5
    "EP3960860NWA2",   # 6
    "EP4141116NWA1",   # 7
    "EP2373382NWB1",   # 8
    "EP4385568NWA2",   # 9
    "EP4744669NWA2",   # 10
]

print(f"IDs consumed by Sections 2 to 4 : {IDS_CSV}")
print(f"Full-text corpus               : {IDS_ALNYLAM}")
print(f"Pilot patents                  : {len(PILOT)}")


## 1. Patent identifier extraction

**Module:** `epo/search.py`, entry point `download_patent_ids`.
**Reads:** EPO OPS (live).  **Writes:** `EPO_siRNA_IDs_<start>_<end>_codes_only.csv`, `..._terms_only.csv`, `..._only_applicant_Alnylam.csv`.

This first stage collects patent **family identifiers** only. No metadata, abstracts or full texts are requested yet, so the download stays small and gives a cheap checkpoint before the expensive sections. Each output row holds `Patent_ID`, `Family_ID` and `Country`.

### One module, three searches

The same invention can be found in different ways, and each way misses something different. That is why the search method is an argument rather than a separate script:

| `strategy` | How it searches | Strength | Blind spot |
|---|---|---|---|
| `"codes"` | CPC and IPC classification codes, 6 queries | Precision: an examiner decided this document is about RNA interference | Recent applications that are not classified yet |
| `"terms"` | siRNA wording in title and abstract, 27 queries | Recall: catches a document the day it publishes | False positives, for example a patent that only mentions siRNA as prior art |
| `"applicant"` | Company name only, subject ignored | One company's complete portfolio, used as a benchmark | Nothing outside that company |
| `"codes+terms"` | Both query sets in one pass | One merged corpus | Cannot be used to compare the two approaches, since it blends them |

The three cells below run the first three. `applicant_filter` is separate from the strategy: passing it with `"codes"` or `"terms"` narrows that subject search to one company, while `strategy="applicant"` makes the company name the whole query.

### Why each code and each term is its own query

The OPS search service never returns more than 2000 results for a query, and the surplus is dropped without warning. Sending 6 or 27 narrow queries instead of one wide one keeps each response under that ceiling, and the heavy overlap between them is resolved afterwards by family deduplication.

A query that still exceeds 2000 for a whole year is sliced automatically: first into twelve monthly windows, then into daily windows for any month still over the limit. A count-only request decides when to slice, so nothing is downloaded to find out.

```text
query for 2024      -> 1 340 results -> fetched directly
query for 2023      -> 4 900 results -> split into 12 months
   └── March 2023   -> 2 600 results -> split into 31 days
```

### Changing what is searched for

Everything specific to siRNA lives in section 1 of `search.py`, right below the imports: `CPC_CODES`, `IPC_CODES`, `EXCLUDED_CPC_CODES` and `SEARCH_TERMS`. Nothing further down that file mentions siRNA.

> **Output filenames encode the years.** `download_patent_ids` builds the output name from `start_year` and `end_year`, so changing the years changes the filename every later section has to read. As written, the code and Alnylam searches cover 2022 to 2025, while the keyword search covers 2001 to 2026 and therefore writes `EPO_siRNA_IDs_2001_2026_terms_only.csv`. Which file the later sections consume is set by `IDS_CSV` in the Run configuration cell, so a search can be re-run over new years without breaking Sections 2 to 4.

### 1a. Strategy A, CPC/IPC classification codes

This search queries the OPS search service by Cooperative and International Patent Classification code. These codes are assigned by examiners, so they are a structured and fairly reliable signal of technical content. The main anchor for siRNA is `C12N 15/113` (RNA interference and small interfering RNA), and the CPC queries deliberately start one level higher, at `C12N 15/11` with the `/low` operator, because the RNAi subgroups only exist since 2006 and the parent code catches the pioneering filings from before that date.

Results are deduplicated at family level using a country priority table (EP first, then US and WO, then the remaining offices), so each family contributes exactly one row.

The call below uses `CONSUMER_KEY` and `CONSUMER_SECRET` from the Credentials cell. The optional quota check prints how much of the weekly 4 GB free-tier allowance has been used so far.


In [ ]:
from sirna_pipeline.epo import search

# Optional quota check: current weekly data use vs the 4 GB free-tier limit
print("Checking EPO OPS quota...")
search.check_epo_quota(CONSUMER_KEY, CONSUMER_SECRET)

# Strategy A: queries built from CPC_CODES and IPC_CODES, every applicant
print("\nStarting CPC/IPC extraction...")
df_codes = search.download_patent_ids(
    consumer_key     = CONSUMER_KEY,
    consumer_secret  = CONSUMER_SECRET,
    start_year       = 2022,
    end_year         = 2025,
    strategy         = "codes",
    applicant_filter = None,      # None = no applicant restriction
)
print(df_codes.head())


### 1b. Strategy B, title and abstract keywords

The second search matches free-text terms in the title and abstract fields (the CQL `ta=` parameter): `siRNA`, `small interfering RNA`, `RNA interference`, `RNAi`, plus the "ribonucleic acid" spellings and other duplex variants, 27 terms in all, one query per term. Restricting the search to title and abstract stops incidental mentions deep in the body text from inflating the results.

It complements Strategy A in two ways. It catches patents that are clearly about siRNA in their wording but have not yet received the `C12N 15/113` code, which is common for recent applications that have not been examined yet. It also catches patents filed under a broader parent code. Because it ignores examiner classification entirely, it provides an independent recall signal.

Query slicing, rate limiting and family deduplication work exactly as in Strategy A. Only the `strategy` argument changes.


In [ ]:
# Strategy B: queries built from SEARCH_TERMS, every applicant, longer period
print("Starting keyword (title/abstract) extraction...")
df_terms = search.download_patent_ids(
    consumer_key     = CONSUMER_KEY,
    consumer_secret  = CONSUMER_SECRET,
    start_year       = 2001,
    end_year         = 2026,
    strategy         = "terms",
    applicant_filter = None,
)
print(df_terms.head())


### 1c. Reference corpus, the full Alnylam Pharmaceuticals portfolio

As an external validation set, the complete Alnylam portfolio is retrieved by querying the applicant-name field directly (`strategy="applicant"`, `applicant_filter="Alnylam*"`). This bypasses all classification and keyword constraints, and the wildcard matches the different Alnylam entity names used across filings.

Alnylam is the leading siRNA company and the originator of the first approved siRNA therapeutics (patisiran, givosiran, lumasiran, inclisiran, vutrisiran), so nearly every family in its portfolio is siRNA relevant. That makes the portfolio a high-confidence benchmark for recall: any Alnylam family missing from the output of Strategy A or B is a relevant family that strategy failed to find.

In [ ]:
# Alnylam full portfolio: applicant-name query, no CPC/IPC or keyword filter.
# Writes IDS_ALNYLAM, the corpus Section 4 downloads full text for.
print("Extracting Alnylam full portfolio...")
df_alnylam = search.download_patent_ids(
    consumer_key     = CONSUMER_KEY,
    consumer_secret  = CONSUMER_SECRET,
    start_year       = 2022,
    end_year         = 2025,
    strategy         = "applicant",   # name-driven query, subject filters ignored
    applicant_filter = "Alnylam*",    # wildcard matches all Alnylam entity names
)
print(df_alnylam.head())


## 2. Bibliographic metadata enrichment

**Reads:** `IDS_CSV`, one of the ID CSVs from Section 1.  **Writes:** `METADATA_CSV`, the same filename with `_metadata` appended.

Section 1 produced lean ID lists. This stage adds the full bibliographic record for each family through the OPS `/biblio` endpoint.

Fields retrieved: earliest priority date, publication date, applicant names, invention title (English preferred, with a fallback), abstract (English preferred), IPC codes and CPC codes.

How the stage protects a long run:

- IDs are sent in batches of 100, which is the `/biblio` hard limit.
- A failing batch is retried up to three times, and then each ID in it is retried on its own, so one bad record cannot cost the other 99.
- If a record comes back with no abstract, a second call to `/abstract` is made for that ID alone.
- An adaptive rate limiter lengthens the pause between batches when the server signals congestion, and triggers a session cooldown after repeated throttling.
- A final pass deduplicates by `Family_ID`, preferring records with a usable English abstract and a Latin-script title, then the oldest priority date.


In [ ]:
from sirna_pipeline.epo.biblio import fetch_biblio_from_csv

df_metadata = fetch_biblio_from_csv(
    ids_csv         = IDS_CSV,
    consumer_key    = CONSUMER_KEY,
    consumer_secret = CONSUMER_SECRET,
)
df_metadata.head()



=== STARTING EPO METADATA FETCH ===
[INFO] Input file : EPO_siRNA_IDs_2022_2025_terms_only.csv
[INFO] Patent IDs : 7405
[INFO] Batches : 75 x 100 IDs per batch

[AUTH] Generating a new EPO access token...
[INFO] Batch 1 complete — 100 records fetched so far.
[INFO] Batch 2 complete — 200 records fetched so far.
  [WARNING] Abstract fallback failed for EP.4381069.A1: ReadTimeout: HTTPSConnectionPool(host='ops.epo.org', port=443): Read timed out. (read timeout=15)
[INFO] Batch 3 complete — 300 records fetched so far.

[AUTH] Generating a new EPO access token...
[INFO] Batch 4 complete — 400 records fetched so far.
[INFO] Batch 5 complete — 500 records fetched so far.
[INFO] Batch 6 complete — 600 records fetched so far.
[INFO] Batch 7 complete — 700 records fetched so far.
[INFO] Batch 8 complete — 800 records fetched so far.
[INFO] Batch 9 complete — 900 records fetched so far.
[INFO] Batch 10 complete — 1000 records fetched so far.
[INFO] Batch 11 complete — 1100 records fetched so fa

,Patent_ID,Country,Number,Kind,Family_ID,Priority_Date,Publication_Date,Applicant,Title,Abstract,IPCs,CPCs
1550,US2022275366A1,US,2022275366,A1,83006944,20010518,20220901,SIRNA THERAPEUTICS INC [US] | SIRNA THERAPEUTI...,RNA INTERFERENCE MEDIATED INHIBITION OF GENE E...,The present invention concerns methods and rea...,,"A61K38/00, A61K47/54, A61K47/544, A61K47/549, ..."
420,US2022056441A1,US,2022056441,A1,53183180,20020220,20220224,SIRNA THERAPEUTICS INC [US] | SIRNA THERAPEUTI...,RNA INTERFERENCE MEDIATED INHIBITION OF GENE E...,The present invention concerns methods and rea...,,"C07H21/02, C12N15/111, C12N15/113, C12N15/1131..."
431,US2022112494A1,US,2022112494,A1,32046073,20020925,20220414,UNIV MASSACHUSETTS [US] | UNIVERSITY OF MASSAC...,IN VIVO GENE SILENCING BY CHEMICALLY MODIFIED ...,The present invention provides compositions fo...,,"A01K2217/075, A61K38/00, A61K48/00, C07D213/69..."
437,US2022315922A1,US,2022315922,A1,50773796,20021114,20221006,THERMO FISHER SCIENTIFIC INC [US] | Thermo Fis...,Methods and Compositions for Selecting siRNA o...,Efficient sequence specific gene silencing is ...,,"A61K31/713, A61K48/00, C12N15/1048, C12N15/111..."
682,US2022062286A1,US,2022062286,A1,27772701,20030725,20220303,UNIV SHEFFIELD [GB] | The University of Sheffield,USE OF RNAI INHIBITING PARP ACTIVITY FOR THE M...,The present invention relates to the use of an...,,"A61K31/472, A61K31/517, A61K31/5517, A61K31/70..."


## 3. Classification into tiers

**Reads:** a metadata CSV from Section 2.  **Writes:** `..._metadata_filtered.csv`.

`filtering.tiers.apply_filters` sorts every patent into one of eight tiers (1, 2, 3, 4A, 4B, 5, 6, 7) from two signals: siRNA wording in the title and abstract, and the structural `C12N15/113` classification anchor. Tiers are tested in priority order and each patent receives the highest tier it qualifies for, so the tiers never overlap.

Nothing is deleted. Every patent is written out with its tier and a direct Espacenet link, which keeps the dataset complete and makes manual curation transparent. The tiers guide the review, they do not replace it. In the output CSV each tier block is preceded by a blank separator row carrying the tier name, which makes the file easy to read by eye. Filter on the tier column before any further processing.

| Tier | Criterion | Recommended action |
|---|---|---|
| **Tier 1** | siRNA wording **and** the `C12N15/113` anchor | Core dataset, include without further review |
| **Tier 2** | Wording only, anchor absent (not yet classified, or filed under a broader code) | High confidence, include. A missing CPC code is not disqualifying |
| **Tier 3** | Anchor only, no wording (abstract missing, non-English or uninformative) | Check on Espacenet before including |
| **Tier 4A** | siRNA wording or anchor, **plus** a competing-technology term (aptamer, antisense oligonucleotide, CRISPR) | Mixed technology, review to identify the primary one |
| **Tier 4B** | Diagnostic or biomarker language, no therapeutic application | Likely out of scope for a therapeutics analysis |
| **Tier 5** | Agricultural, veterinary or pest-control application **with** the anchor | Depends on scope (RNAi in plants and insects) |
| **Tier 6** | No wording and no anchor | Likely irrelevant, lowest priority |
| **Tier 7** | Agricultural or veterinary application **without** the anchor | Likely irrelevant |


**Known limitation.** Keyword and CPC rules cannot handle negation or ambiguous phrasing. For now the rule-based system gives enough signal to move on to full-text processing at a defensible confidence level.

In [ ]:
from sirna_pipeline.filtering.tiers import apply_filters

result_df = apply_filters(
    raw_data        = METADATA_CSV,   # written by Section 2
    output_filename = FILTERED_CSV,
    csv_sep         = ";",
    csv_encoding    = "utf-8-sig",
)
display(result_df.head(20))


[INFO] Reading CSV from disk: EPO_siRNA_IDs_2022_2025_terms_only_metadata.csv

=== STARTING PATENT CLASSIFICATION ===
[INFO] No patents will be deleted — all records go to the output CSV.

[SUCCESS] Classification complete.
  Total input patents : 7405

  TIER 1 — siRNA Confirmed (Text + CPC)             1588  ███████████████
  TIER 2 — siRNA Confirmed (Text only)              1170  ███████████
  TIER 3 — siRNA by CPC only (Check Abstract)        604  ██████
  TIER 4A — Mixed Tech (siRNA/CPC + Forbidden Term  1369  █████████████
  TIER 4B — Diagnostic/Biomarker only (Review)       145  █
  TIER 5 — Agri/Vet with siRNA CPC (Review)          361  ███
  TIER 6 — No siRNA Signal (Likely Irrelevant)      1872  ██████████████████
  TIER 7 — Agri/Vet without siRNA (Likely Irreleva   296  ██

 There are 1 patent(s) flagged as Needs_Espacenet_Review (unreadable title / missing abstract)

 There are 7 patent(s) flagged with Quantitative Efficacy Data

  Output saved to: EPO_siRNA_IDs_2022_2025_t

,Patent_ID,Priority_Date,Publication_Date,Applicant,Title,Abstract,Espacenet_Link,Tier,Has_Efficacy_Data,Needs_Espacenet_Review,IPCs,CPCs,Family_ID
0,--- TIER 1 — SIRNA CONFIRMED (TEXT + CPC) (158...,,,,,,,,,,,,
1,US2022112494A1,20020925,20220414,UNIV MASSACHUSETTS [US] | UNIVERSITY OF MASSAC...,IN VIVO GENE SILENCING BY CHEMICALLY MODIFIED ...,The present invention provides compositions fo...,https://worldwide.espacenet.com/publicationDet...,TIER 1 — siRNA Confirmed (Text + CPC),,,NaN,"A01K2217/075, A61K38/00, A61K48/00, C07D213/69...",32046073
2,US2022315922A1,20021114,20221006,THERMO FISHER SCIENTIFIC INC [US] | Thermo Fis...,Methods and Compositions for Selecting siRNA o...,Efficient sequence specific gene silencing is ...,https://worldwide.espacenet.com/publicationDet...,TIER 1 — siRNA Confirmed (Text + CPC),,,NaN,"A61K31/713, A61K48/00, C12N15/1048, C12N15/111...",50773796
3,US2022062286A1,20030725,20220303,UNIV SHEFFIELD [GB] | The University of Sheffield,USE OF RNAI INHIBITING PARP ACTIVITY FOR THE M...,The present invention relates to the use of an...,https://worldwide.espacenet.com/publicationDet...,TIER 1 — siRNA Confirmed (Text + CPC),,,NaN,"A61K31/472, A61K31/517, A61K31/5517, A61K31/70...",27772701
4,US2022119814A1,20040709,20220421,UNIV MASSACHUSETTS [US] | University of Massac...,Therapeutic alteration of transplantable tissu...,"The present invention, at least in part, relat...",https://worldwide.espacenet.com/publicationDet...,TIER 1 — siRNA Confirmed (Text + CPC),,,NaN,"A01N1/126, A61K47/6911, A61K48/005, C12N15/111...",36125793
5,US2022315945A1,20050916,20221006,MONSANTO TECHNOLOGY LLC [US] | Monsanto Techno...,Methods for genetic control of insect infestat...,The present invention relates to control of pe...,https://worldwide.espacenet.com/publicationDet...,TIER 1 — siRNA Confirmed (Text + CPC),,,NaN,"C07H21/04, C07K14/43536, C07K14/43563, C12N15/...",37497032
6,US2022042021A1,20051229,20220210,ARROWHEAD PHARMACEUTICALS INC [US] | Arrowhead...,RNAi-MEDIATED INHIBITION OF HIF1A FOR TREATMEN...,RNA interference is provided for inhibition of...,https://worldwide.espacenet.com/publicationDet...,TIER 1 — siRNA Confirmed (Text + CPC),,,NaN,"A61K31/7105, A61K31/713, A61K9/0048, A61P27/00...",38218798
7,US2022112505A1,20070615,20220414,ARROWHEAD PHARMACEUTICALS INC [US] | Arrowhead...,RNAi Inhibition of Alpha-ENaC Expression,The invention relates to compositions and meth...,https://worldwide.espacenet.com/publicationDet...,TIER 1 — siRNA Confirmed (Text + CPC),,,NaN,"A61K31/713, A61K45/06, A61P11/00, A61P11/06, A...",40130244
8,US2023053332A1,20080902,20230223,ALNYLAM PHARMACEUTICALS INC [US] | LUDWIG INST...,COMPOSITIONS AND METHODS FOR INHIBITING EXPRES...,The invention relates to a double-stranded rib...,https://worldwide.espacenet.com/publicationDet...,TIER 1 — siRNA Confirmed (Text + CPC),,,NaN,"A61K31/713, A61P35/00, C12N15/1136, C12N15/113...",41268470
9,US2022243199A1,20080925,20220804,ALNYLAM PHARMACEUTICALS INC [US] | Alnylam Pha...,Lipid formulated compositions and methods for ...,The invention relates to a double-stranded rib...,https://worldwide.espacenet.com/publicationDet...,TIER 1 — siRNA Confirmed (Text + CPC),,,NaN,"A61K31/713, A61P1/00, A61P1/04, A61P1/16, A61P...",41349261


## 4. Full-text XML download

**Reads:** an ID CSV with `Patent_ID` and `Family_ID`.  **Writes:** `eps_xmls/*.xml`, plus `successful_downloads.csv` and `not_in_eps.csv` in the working directory.

This stage downloads full text from the European Publication Server (EPS): description, claims, and the experimental tables that record siRNA activity against target genes. That full text is the input for Section 5.

`epo/fulltext.py` works on the **whole patent family**, not on one publication, because family members are not copies of each other. Divisionals, continuations, and even the A (application) and B (granted) versions of the same application can carry different experimental data. Instead of guessing which relative is best, the module pulls every EP member and leaves the comparison to a later step.

For each patent in the input CSV it:

1. **Fetches the whole family** from EPO OPS, all members and all countries, using both the `/equivalents` service and the `famn=<Family_ID>` family search.
2. **Logs every non-EP member** (US, WO, JP and so on) straight to `not_in_eps.csv`. Only EP publications can have full text on EPS, so these are never requested.
3. **Tests every EP member individually** on EPS, across all seven kind codes (A1, A2, A3, A4, B1, B2, B3) and publication numbers, without stopping at the first hit. Members whose XML has a real full-text structure (`<description>`, `<claims>`, `<table>`) are saved into `eps_xmls/`. Every EP member without full text is written to `not_in_eps.csv` with the reason.

Files already present in `eps_xmls/` are skipped, so an interrupted run can simply be restarted and will fetch only what is missing. A strict 8 second pause follows every EPS and OPS request, which makes a full family sweep deliberately slow.

**Outputs.** `successful_downloads.csv` records what was saved and how each member relates to the requested patent. `not_in_eps.csv` records every member with no EPS full text, which is useful for later coverage analysis.

In [ ]:
from sirna_pipeline.epo.fulltext import download_eps_xmls_with_ops

download_eps_xmls_with_ops(
    csv_filename     = IDS_ALNYLAM,
    consumer_key     = CONSUMER_KEY,
    consumer_secret  = CONSUMER_SECRET,
    output_directory = XML_DIR,
)


Starting FULL-FAMILY extraction for 316 patents...
Enforcing strictly 8+ second delays between all requests.
------------------------------------------------------------

Processing family of: EP4658280A2...
  Family: 3 members (1 EP number(s), 2 non-EP)
  [NO XML] EP4658280 -> no full text on EPS for any kind code (logged).

Processing family of: EP4561631A2...
  Family: 4 members (1 EP number(s), 3 non-EP)
  [NO XML] EP4561631 -> no full text on EPS for any kind code (logged).

Processing family of: EP4594492A1...
  Family: 3 members (1 EP number(s), 2 non-EP)
  [NO XML] EP4594492 -> no full text on EPS for any kind code (logged).

Processing family of: EP4547852A2...
  Family: 4 members (1 EP number(s), 3 non-EP)
  [NO XML] EP4547852 -> no full text on EPS for any kind code (logged).

Processing family of: EP4522742A2...
  Family: 4 members (1 EP number(s), 3 non-EP)
  [NO XML] EP4522742 -> no full text on EPS for any kind code (logged).

Processing family of: EP4547683A2...
  Famil

## 5. Table isolation from the full-text XMLs

**Reads:** `eps_xmls/*.xml`.  **Writes:** `isolated_tables/*.xml`, one file per table.

Each full-text XML is parsed with BeautifulSoup and `lxml` to isolate the experimental tables. `tables/isolate.py` looks for the `EXAMPLES` heading, the standard EPO boundary between the general description and the experimental section, and then takes every top-level `table` or `tables` element that comes after it and sits inside `<description>`. A patent with no `EXAMPLES` heading, or no tables after it, is skipped with a message on screen.

For each table it also copies the **five paragraphs immediately before it**, since those usually carry the assay conditions, the cell line and the setup needed to interpret the numbers. Paragraphs from before the `EXAMPLES` heading or outside the description are dropped, and any table nested inside a copied paragraph is removed so the context holds text only. Table plus context are written as one self-contained XML file.

Output filenames follow `<patent_id>_table_<NN>[_T<num>][_in_vitro].xml`:

- `<patent_id>`: base name of the source XML, for example `EP2723758NWB1`.
- `<NN>`: zero-padded position of the table in the document, which keeps ordering stable and filenames unique.
- `T<num>`: the real table number read from the title, for example `T18b`. Omitted when the title has no recognisable `Table <N>` label.
- `_in_vitro`: added when the title mentions any of *antisense strand, cells, in vitro, sense strand, transfection, single dose, dose response, modified sequences, antisense sequence, sense sequence*. This makes the in vitro tables easy to select later.

`extract_tables_from_patent` also takes `descriptor_words=N`, which appends N descriptor words from the title after the table number, giving names such as `T18b_ic_50_pm`. The default of 0 is recommended: descriptors read as clutter and can cut mid-identifier.

The cell below runs on `PILOT`, the ten patents chosen in the Run configuration cell to cover different table layouts, which is enough to test the LLM steps in Sections 6 and 7 at low cost. To process everything instead, uncomment the `glob` line and use it to build `patent_files`.


In [ ]:
from sirna_pipeline.tables.isolate import extract_tables_from_patent
import glob
import os

# EVERY XML downloaded in Section 4:
# patent_files = sorted(glob.glob(os.path.join(XML_DIR, "*.xml")))

# --- Small test: the pilot set from the Run configuration cell
patent_files = [os.path.join(XML_DIR, f"{pid}.xml") for pid in PILOT]

for path in patent_files:
    print(f"Processing {os.path.basename(path)}...")
    extract_tables_from_patent(path, output_dir=TABLE_DIR)

print(f"\nTable isolation complete - {len(patent_files)} patent file(s) processed.")


Processing EP2723758NWB1.xml...
  Saved: isolated_tables\EP2723758NWB1_table_01_T1.xml
  Saved: isolated_tables\EP2723758NWB1_table_02_T2_in_vitro.xml
  Saved: isolated_tables\EP2723758NWB1_table_03_T3_in_vitro.xml
  Saved: isolated_tables\EP2723758NWB1_table_04_T4_in_vitro.xml
  Saved: isolated_tables\EP2723758NWB1_table_05_T5_in_vitro.xml
  Saved: isolated_tables\EP2723758NWB1_table_06_T6.xml
  Saved: isolated_tables\EP2723758NWB1_table_07_T7_in_vitro.xml
  Saved: isolated_tables\EP2723758NWB1_table_08_T8_in_vitro.xml
  Saved: isolated_tables\EP2723758NWB1_table_09_T9_in_vitro.xml
  Saved: isolated_tables\EP2723758NWB1_table_10_T10_in_vitro.xml
  Saved: isolated_tables\EP2723758NWB1_table_11_T11_in_vitro.xml
  Saved: isolated_tables\EP2723758NWB1_table_12_T12_in_vitro.xml
  Saved: isolated_tables\EP2723758NWB1_table_13_T13_in_vitro.xml
  Saved: isolated_tables\EP2723758NWB1_table_14_T14_in_vitro.xml
  Saved: isolated_tables\EP2723758NWB1_table_15_T15.xml
Processing EP2999785NWB1.xml.

## 6. XML to CSV, with header normalisation

**Reads:** `isolated_tables/`.  **Writes:** `csv_output/<base>_tables.csv`, `csv_output/<base>_context.txt`, `csv_output/debug/`.

`tables/parse.py` turns each isolated table into a structured CSV. It solves two problems at once: pulling the data out of CALS-style XML, and turning the heterogeneous, often multi-level headers of patent tables into clean SQL identifiers. Groq keys from the Credentials cell are rotated automatically to stay inside free-tier limits.

Each input XML produces two files:

- `<base>_context.txt`: the table title and the context paragraphs kept in Section 5, plus any full-width annotation rows (method notes, footnotes, spanning captions) that are not column names.
- `<base>_tables.csv`: the table data with normalised headers.

### Structural repairs, before anything reaches the LLM

Patent drafters break the CALS conventions constantly, and a reader that trusts the markup produces a broken CSV. Five recurring problems are handled:

1. **Column names stored as data.** A `tbody` row is promoted to a header when it is one cell wide across the whole table, or when every one of its cells spans a range of columns. A row that mixes ordinary cells with one spanning cell is left alone, because that is a normal two-level header.
2. **Header split across `thead` and `tbody`.** When the group labels sit in `thead` and the leaf labels in the first `tbody` row, that row has ordinary cells and repair 1 misses it. It is promoted only when every conservative check passes: no spanning header was found, at least two rows remain, every non-empty cell in the row is non-numeric, the first cell does not look like a duplex or compound ID, and at least one cell just below it is numeric.
3. **Blank group labels.** In CALS a label covering several rows is written once and the cells below it are left blank. The last non-empty value in column 0 is carried down into any later row that is blank there but has data elsewhere, which restores the cell line on every row.
4. **Footnotes that look like a table.** A trailing `tgroup` with no `thead` whose rows are all full-width text has no columns and no data, so its text goes to the context file. Full-width rows inside a normal table are sorted by length: a short one such as `HeLa day 3` is a section divider, and when a table stacks two or more labelled sections the label is kept as a trailing `section` column so the condition is not lost. A long one ending in a period is a caption and goes to the context file.
5. **OCR damage in concentration labels.** Scanned headers turn `0.1nM` into `O.lnM`. Only the numeric part immediately before `nM` is repaired, so ordinary text is never altered.

Tables whose title contains "abbreviation" are skipped entirely.

### Header normalisation

**The data is the ground truth for the column count.** Every data row in a CALS table has one cell per column, so the header must end up with exactly that many names. Any other count means a column was dropped, added or reordered, which silently misaligns every value in the table. The whole strategy is built around that check.

1. **Deterministic merge** (`merge_multilevel_headers`) runs first and owns the grid. It concatenates the header rows column by column, so it can never drop, add or reorder a column, and exact consecutive duplicates within a column are skipped so a `morerows` cell does not produce `Duplex ID Duplex ID`. It also keeps two identical `STDEV` columns apart, since each inherits a different group label. If the merged width equals the data width it is accepted and **no LLM call is made for that table**.
2. **LLM re-fusion, only on a mismatch** (`repair_headers_with_ai`, `llama-3.3-70b-versatile`). When the spans cannot account for every data column, the model is given the header grid, the table title and up to four sample rows, and returns one flat list of column names. It is accepted only if its length also matches the data width.
3. **Unresolved.** If neither method aligns, no shifted header is shipped. The table gets positional names `_col_0`, `_col_1`, and so on, a `[WARN]` is printed and the reason is written to the debug log for manual review.
4. **SQL names** (`normalize_headers_with_ai`, `llama-3.1-8b-instant`, printed on screen as `[Pass 2]`). The clean human-readable strings become SQL identifiers: lowercase, underscores, `%` to `_pct`, `#` to `_num`, brackets removed but their contents kept, `5'` and `3'` apostrophes dropped, unit-only headers given a `conc_` prefix, plus siRNA-specific mappings such as `IC50 (nM)` to `ic50_nm`. The call is batched over the unique headers of the whole file, not per table, to save tokens against the Groq TPM limit. The rule-based `basic_sql_normalize` takes over if the API cannot be reached.

**Deterministic fixes after the LLM**, each one added because the model was seen to get that case wrong: day and time qualifiers are re-normalised by rule, so `Day 3 1nM` becomes `day_3_1nm` instead of collapsing three timepoints into one name; every column is scanned and the first whose values are mostly `AD-\d+` becomes `duplex_id`, since that column is not always the first one; any SQL name containing `duplex` becomes `duplex_id`; and duplicate names get their column index appended (`stdev`, `stdev_4`).

**Avg/SD plausibility check.** A width check catches a dropped or duplicated column but not a swap, where every count still matches while the values are wrong. Paired `*_avg` and `*_sd` columns are therefore compared, and a warning is logged when the SD median exceeds the Avg median. It only warns and never edits the data, so a false positive is harmless.

**Debug trail.** `csv_output/debug/` holds one session log per run and one log per file: which header path was taken for each table, the raw LLM input and output, and every fix applied.

**Optional audit file.** With `create_headers_file=True`, a `<base>_llm_normalize.py` is written next to each source XML. It is runnable standalone, encodes the exact mapping rules the LLM applied, and exits non-zero if any mapping does not hold, which catches an LLM shortcut at generation time.

> **Model IDs may need updating.** Groq lists both `llama-3.3-70b-versatile` and `llama-3.1-8b-instant` as deprecated, with a decommission date of 2026-08-16 noted in the source. The IDs are set at the top of `tables/parse.py` (this section) and `assembly/core.py` (Section 7), so check the Groq deprecations page before starting a long run.


In [ ]:
from sirna_pipeline.tables.parse import convert_directory

convert_directory(
    TABLE_DIR,
    output_dir          = CSV_DIR,
    api_keys            = GROQ_API_KEYS,
    create_headers_file = False,   # True also writes <base>_llm_normalize.py beside each XML
)


  [Groq] 4 API key(s) loaded.

Processing: EP2373382NWB1_table_01_T1.xml
  -> EP2373382NWB1_table_01_T1_context.txt  (4 paragraph(s))
  SKIP tables file (no data tables found in EP2373382NWB1_table_01_T1.xml)

Processing: EP2373382NWB1_table_02.xml
  [Pass 2] Normalising 2 unique header(s) to SQL via Groq …
  -> EP2373382NWB1_table_02_context.txt  (4 paragraph(s))
  -> EP2373382NWB1_table_02_tables.csv  (1 table(s))

Processing: EP2373382NWB1_table_03_T2a.xml
  [Pass 2] Normalising 4 unique header(s) to SQL via Groq …
  -> API error: LLM response contained no JSON object.
  [Groq] All attempts exhausted. Using rule-based fallback.
  -> EP2373382NWB1_table_03_T2a_context.txt  (7 paragraph(s))
  -> EP2373382NWB1_table_03_T2a_tables.csv  (1 table(s))

Processing: EP2373382NWB1_table_04_T2b_in_vitro.xml
  [Pass 2] Normalising 4 unique header(s) to SQL via Groq …
  -> EP2373382NWB1_table_04_T2b_in_vitro_context.txt  (7 paragraph(s))
  -> EP2373382NWB1_table_04_T2b_in_vitro_tables.csv  (1 ta

## 7. Primary table assembly

**Reads:** `csv_output/`.  **Writes:** four numbered folders beside `OUTPUT_CSV`, listed at the end of this section.

The last stage consolidates the per-table CSVs into three fixed schemas. For each input CSV, `assembly/build.py` asks an LLM (`llama-3.3-70b-versatile`) to write a DuckDB `SELECT` that maps that file's particular columns onto the target schema. DuckDB then runs the query locally, so the model writes the mapping and never touches the data values. A no-fabricated-values guard checks that every measurement column in the generated SQL really refers to a column of the CSV.

**How a table becomes rows**

1. **Routing** (`routing.py`) decides what each table measures, from the paired `_context.txt` and the headers: knockdown, IC50 or viability. An LLM classification, backed by an on-disk decision cache, runs alongside deterministic checks, and a table can carry more than one measurement type. Immune-response and in-vivo tables, and tables whose content contradicts their route, are flagged and their rows quarantined in `flagged_rows` rather than dropped.
2. **Mapping** (`sql_builder.py`) builds the prompt for the target schema, including detectors for sparse sequence layouts where one duplex ID appears once and applies to several rows.
3. **Execution.** DuckDB runs that SQL locally against the CSV, so no value is ever produced by the model.
4. **Validation** (`core.py`). Numeric fields must be numeric, dose and IC50 must sit between 0 and 10^7 nM (10 mM), inhibition must sit between -200 % and 200 %, since a negative value is genuine upregulation and must be kept, and sequence fields must look like sequences. A failing cell is blanked and recorded in `validation_failures` rather than silently corrupting the table.

| Output file | Content | Key fields |
|---|---|---|
| `primary_table.csv` | Knockdown activity (percent inhibition at a dose) | `duplex_id`, `sense_sequence`, `antisense_sequence`, `sense_oligo_id`, `antisense_oligo_id`, `cell_line`, `dose_nM`, `inhibition_percent`, `value_sd`, `replicate`, `transfection_method`, `target_gene_name` |
| `primary_ic50_table.csv` | IC50 values, with replicate and timepoint detail | `duplex_id`, `cell_line`, `timepoint_hrs`, `replicate`, `ic50_nM`, `ic50_unit_original`, `transfection_method`, `target_gene_name` |
| `primary_cell_viability_table.csv` | Cell-viability screens | `duplex_id`, `cell_line`, `day`, `dose_nM`, `viability_value`, `viability_sd`, `viability_basis`, `viability_relative_to`, `transfection_method`, `target_gene_name` |

Every row also carries `patent_id` and `source_file`. Viability numbers are stored exactly as the patent reports them and are never rescaled: `viability_basis` (a six-value vocabulary, `unknown` when the patent does not say) and `viability_relative_to` record what the value is relative to, for example a non-targeting control such as AD-1955.

**Merging (knockdown table only).** Rows sharing `(patent_id, duplex_id, cell_line, dose_nM)` are merged into one. Annotations (sequences, oligo IDs, target gene) and the measurement fields take the first non-null value, and `source_file` accumulates every contributing filename. Rows carrying only sequences or oligo IDs, with no measurement, are used to enrich matching activity rows and are then dropped, so the output contains no annotation-only records. Both sequence forms are kept: `sense_sequence` and `antisense_sequence` hold the modified form when available, and `*_sequence_unmodified` hold the plain form. The IC50 and viability tables are not merged, since each of their rows is an independent condition.

**Resilience.** The SQL generated for each table is cached on disk under `4_trace/sql_cache/`, keyed by a hash of the table content and the prompt. If a run stops, for example on a sustained rate limit, restarting reuses the cached SQL for finished tables and calls the API only for the rest. Deleting that folder forces a clean regeneration.

**Output layout.** Everything is written under the folder of `output_path`, one subfolder per stage of the run, so the finished data, the review pile, the drafts and the decision trail never sit together:

| Folder | Content |
|---|---|
| `1_final_tables/` | the dataset: `primary_table*.csv`, `primary_ic50_table*.csv`, `primary_cell_viability_table*.csv` |
| `2_review/` | `failed_tables*.csv`, `validation_failures*.csv`, `flagged_rows*.csv` |
| `3_per_file_drafts/` | each input file's extraction, before merging |
| `4_trace/` | session log, per-table logs, gene log, `sql_cache/` |

The `per_file_dir` argument still works but is only kept for backwards compatibility. Leave it out, as below, and the drafts land in `3_per_file_drafts/` with everything else.


In [ ]:
from sirna_pipeline.assembly.build import build_primary_table

build_primary_table(
    CSV_DIR,
    output_path   = OUTPUT_CSV,    # its folder becomes the run root (4 numbered subfolders)
    api_keys      = GROQ_API_KEYS,
    file_prefixes = PILOT,         # the same ten patents isolated in Section 5
)


  [Groq] 4 API key(s) loaded.
Found 14 table file(s) across 1 group(s) in 'csv_output'.

[1/1] EP2723758NWB1 — 14 tables

[EP2723758NWB1 table 1/14] Processing: EP2723758NWB1_table_02_T2_in_vitro_tables.csv
  Table type: primary  (measurements=none, via llm)
  Generating SQL...
  Executing SQL via DuckDB...
  → 62 row(s)
  → per-file CSV: per_file_output\EP2723758NWB1_table_02_T2_in_vitro_tables_primary.csv

[EP2723758NWB1 table 2/14] Processing: EP2723758NWB1_table_03_T3_in_vitro_tables.csv
  Table type: primary  (measurements=none, via llm)
  Generating SQL...
  Executing SQL via DuckDB...
  → 62 row(s)
  → per-file CSV: per_file_output\EP2723758NWB1_table_03_T3_in_vitro_tables_primary.csv

[EP2723758NWB1 table 3/14] Processing: EP2723758NWB1_table_04_T4_in_vitro_tables.csv
  Table type: primary  (measurements=['knockdown'], via llm)
  Generating SQL...
  Executing SQL via DuckDB...
  → 128 row(s)
  → per-file CSV: per_file_output\EP2723758NWB1_table_04_T4_in_vitro_tables_primary.csv

## Summary

This notebook implements a reproducible route from the EPO patent corpus to a structured database of siRNA activity data, in seven stages:

1. **Identifier extraction:** one module, `epo/search.py`, run three times with a different `strategy`: classification codes, title and abstract keywords, and an applicant-name query for the Alnylam reference portfolio.
2. **Metadata enrichment:** full bibliographic records through the OPS `/biblio` endpoint, with per-ID retries and an `/abstract` fallback.
3. **Tier filtering:** an eight-tier rule-based classification that guides curation without discarding any record.
4. **Full-text XML download:** the whole EP family for each patent from EPS, probed across all seven kind codes, with every member lacking full text logged to `not_in_eps.csv`.
5. **Table isolation:** the experimental tables, plus their assay context, lifted out of the EXAMPLES sections.
6. **XML to CSV:** structural repairs to the CALS markup, then headers normalised into consistent SQL names by a deterministic merge that owns the column grid, with the LLM called only when the merge cannot account for every data column.
7. **Primary table assembly:** the per-table CSVs consolidated into knockdown, IC50 and viability tables through LLM-generated DuckDB SQL, with routing, merging, caching and validation.

Throughout the two LLM stages the model only ever reads column headers and context and writes the mapping. Every value in the final dataset is executed out of the patent by DuckDB, not generated by a model.

The Section 7 run root holds `1_final_tables/` (the dataset), `2_review/` (what to check by hand), `3_per_file_drafts/` (per-file extractions before merging) and `4_trace/` (how it decided).
